# 13 · Video: pronóstico de AIFS para John (2024) frente a lo observado

Un pronóstico de 72 h de AIFS Single v1.1 (inicializado con ERA5) junto a la lluvia observada de MSWEP V3,
cada 6 h, con la trayectoria de IBTrACS. Salida: MP4 de 1920 × 1080 y un PNG por plazo.

Archivos que necesita (copiados de Curnagl a D:):
- `aifs11_regional_<AAAAMMDDHH>.nc` del inicio elegido (13 tiempos: plazo 0 a 72 h)
- `ibtracs.EP.list.v04r01.csv`
- `mswep_v3_6h_2024.nc` (ya está en `/mnt/d/MSWEP_V316_6H/`)

La lluvia de 6 h de AIFS en el plazo *L* acumula de *L* − 6 a *L* h; se compara con MSWEP en el mismo tiempo
válido (sin desfase, ver `scripts/block2/check_tp_window.py`). El plazo 0 no tiene lluvia y se omite.

In [ ]:
# 1. Configuración
import shutil, subprocess
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

from itchi.plotting import MapStyle, configure_map_axis, get_crameri_colormap

PC = ccrs.PlateCarree()
INICIO = "2024092300"                             # probar también 2024092200 y 2024092212
DIR_AIFS = Path("/mnt/d/AITCHazard/aifs_john2024")
IBTRACS = Path("/mnt/d/AITCHazard/ibtracs.EP.list.v04r01.csv")
MSWEP = Path("/mnt/d/MSWEP_V316_6H/mswep_v3_6h_2024.nc")
NOMBRE, TEMPORADA = "JOHN", 2024
LON, LAT = (-110.0, -88.0), (10.0, 22.0)          # 22° × 12°, casi 16:9 para un solo panel
LLUVIA_MAX = 100.0                                # mm en 6 h, tope de la escala (fija para todos los cuadros)
ANCHO, ALTO, FPS = 1920, 1080, 30
PAUSA_S, FUNDIDO_S, BUCLE = 1.2, 0.4, False
OUT = Path("outputs/figures/aitchazard/video_john2024"); OUT.mkdir(parents=True, exist_ok=True)
MESES = {5: "may", 6: "jun", 7: "jul", 8: "ago", 9: "sep", 10: "oct", 11: "nov"}
KT = 1.943844
plt.rcParams.update({"font.size": 14, "axes.titlesize": 18})

def fecha(t):
    return f"{t.day} {MESES[t.month]} {t:%H} UTC"

In [ ]:
# 2. Datos
ini = pd.Timestamp(INICIO[:8] + " " + INICIO[8:] + ":00")
fa = xr.open_dataset(DIR_AIFS / f"aifs11_regional_{INICIO}.nc", decode_timedelta=False)
la_n = "latitude" if "latitude" in fa.coords else "lat"
lo_n = "longitude" if "longitude" in fa.coords else "lon"
lon_a, lat_a = fa[lo_n].values, fa[la_n].values
if np.issubdtype(fa["time"].dtype, np.datetime64):
    validos = pd.DatetimeIndex(fa["time"].values)
else:
    validos = ini + pd.to_timedelta(6 * np.arange(fa.sizes["time"]), unit="h")
tp = fa["tp"].values.astype(float)
if np.nanmax(tp) < 1.0:                           # metros → mm
    tp = tp * 1000.0
viento = np.hypot(fa["u10"].values, fa["v10"].values)
print("AIFS:", dict(fa.sizes), "| válidos", validos[0], "→", validos[-1],
      "| tp máx (mm/6 h) por plazo:", np.round(np.nanmax(tp, axis=(1, 2)), 0))

ms = xr.open_dataset(MSWEP)["precipitation"]
ms_la = "lat" if "lat" in ms.coords else "latitude"
ms_lo = "lon" if "lon" in ms.coords else "longitude"
ms = ms.where((ms[ms_lo] >= LON[0] - 1) & (ms[ms_lo] <= LON[1] + 1)
              & (ms[ms_la] >= LAT[0] - 1) & (ms[ms_la] <= LAT[1] + 1), drop=True)
print("MSWEP recorte:", dict(ms.sizes))

ib = pd.read_csv(IBTRACS, skiprows=[1], low_memory=False)
ib = ib[(ib["NAME"] == NOMBRE) & (ib["SEASON"] == TEMPORADA)].copy()
ib["t"] = pd.to_datetime(ib["ISO_TIME"])
ib[["LAT", "LON"]] = ib[["LAT", "LON"]].astype(float)
ib = ib.set_index("t")
print("IBTrACS:", ib.index.min(), "→", ib.index.max(), "|", len(ib), "registros")

In [ ]:
# 3. Cuadros: un panel con la lluvia pronosticada por AIFS
VIENTO = True                                     # contornos de 34 y 64 kt
TRAYECTORIA = True                                # trayectoria y posición de IBTrACS
cmap = get_crameri_colormap("oslo", reverse=True)
norm = mcolors.LogNorm(1.0, LLUVIA_MAX)
estilo = MapStyle(lon_extent=LON, lat_extent=LAT, lon_tick_spacing=2.0, lat_tick_spacing=2.0, grid_spacing=2.0,
                  label_size=12.0, coastline_linewidth=1.0, border_linewidth=0.8)

def cuadro(i):
    t = validos[i]
    fig, ax = plt.subplots(figsize=(ANCHO / 100, ALTO / 100), dpi=100, subplot_kw={"projection": PC},
                           layout="constrained", facecolor="#FBFAF7")
    configure_map_axis(ax, crs=PC, style=estilo)
    ax.pcolormesh(lon_a, lat_a, np.where(tp[i] >= 1.0, tp[i], np.nan), cmap=cmap, norm=norm, transform=PC,
                  rasterized=True, zorder=2)
    ley = []
    if VIENTO:
        ax.contour(lon_a, lat_a, viento[i] * KT, levels=[34, 64], colors=["#A84B1C", "#5B1A00"],
                   linewidths=[1.4, 2.2], transform=PC, zorder=7)
        ley += [Line2D([], [], color="#A84B1C", lw=1.4, label="Viento 10 m ≥ 34 kt"),
                Line2D([], [], color="#5B1A00", lw=2.2, label="Viento 10 m ≥ 64 kt")]
    if TRAYECTORIA:
        ax.plot(ib["LON"], ib["LAT"], "-", color="0.35", lw=1.2, transform=PC, zorder=6)
        if t in ib.index:
            ax.plot(ib.loc[t, "LON"], ib.loc[t, "LAT"], "*", ms=18, mfc="white", mec="k", mew=1.4,
                    transform=PC, zorder=8)
        ley += [Line2D([], [], color="0.35", lw=1.2, marker="*", ms=10, mfc="white", mec="k",
                       label="IBTrACS (observado)")]
    if ley:
        ax.legend(handles=ley, loc="lower left", fontsize=12, framealpha=0.9)
    fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.8, aspect=30,
                 label="Lluvia pronosticada en 6 h (mm)", extend="max")
    lead = int(round((t - ini) / pd.Timedelta(hours=1)))
    ax.set_title(f"AIFS · John (2024) · inicio {fecha(ini)} · plazo +{lead} h · válido {fecha(t)}",
                 loc="left", fontsize=20, weight="bold")
    fig.canvas.draw()
    rgb = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    fig.savefig(OUT / f"{INICIO}_plazo{lead:02d}.png", dpi=100, facecolor=fig.get_facecolor())
    plt.close(fig)
    return rgb

cuadros = [cuadro(i) for i in range(1, len(validos))]          # sin plazo 0
print(len(cuadros), "cuadros", cuadros[0].shape)

In [ ]:
# 4. Video
ff = shutil.which("ffmpeg")
assert ff, "falta ffmpeg (sudo apt install -y ffmpeg)"
h, w = cuadros[0].shape[:2]
h2, w2 = h - h % 2, w - w % 2                      # libx264 pide dimensiones pares
mp4 = OUT.parent / f"aifs_john2024_{INICIO}.mp4"
p = subprocess.Popen([ff, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{w2}x{h2}",
                      "-r", str(FPS), "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", str(mp4)],
                     stdin=subprocess.PIPE)
sec = [c[:h2, :w2] for c in cuadros] + ([cuadros[0][:h2, :w2]] if BUCLE else [])
n_p, n_f = int(PAUSA_S * FPS), int(FUNDIDO_S * FPS)
for a, b in zip(sec[:-1], sec[1:]):
    for _ in range(n_p):
        p.stdin.write(a.tobytes())
    for x in np.linspace(0, 1, n_f, endpoint=False)[1:]:
        p.stdin.write(((1 - x) * a + x * b).astype(np.uint8).tobytes())
if not BUCLE:
    for _ in range(2 * n_p):                       # el último plazo se queda un poco más
        p.stdin.write(sec[-1].tobytes())
p.stdin.close(); p.wait()
print("video:", mp4, "| tamaño", f"{w2}x{h2}")